## 1

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Production ETL Job") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/06 14:45:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 4.1.0


## 2

In [3]:
from urllib.request import urlretrieve

url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet'
file = './data/yellow_tripdata_2025-11.parquet'
urlretrieve(url, file)

('./data/yellow_tripdata_2025-11.parquet',
 <http.client.HTTPMessage at 0x7480b57b5280>)

In [4]:
df = spark.read.parquet(file)

In [3]:
repartitioned_path = './data/repartitioned/'

In [5]:
df.repartition(4).write \
    .mode("overwrite") \
    .parquet(repartitioned_path)

## 3

In [4]:
dfre = spark.read.parquet(repartitioned_path)

In [5]:
import pyspark.sql.functions as F

dfre \
    .select() \
    .filter(F.to_date(F.col('tpep_pickup_datetime')) == '2025-11-15') \
    .count()

162604

## 4

In [30]:
dfre \
    .withColumn('DiffInHours', F.timestamp_diff('minute', 'tpep_pickup_datetime', 'tpep_dropoff_datetime') / 60 ) \
    .select(F.max('DiffInHours')) \
    .show(truncate=False)

+-----------------+
|max(DiffInHours) |
+-----------------+
|90.63333333333334|
+-----------------+



## 6

In [33]:
from urllib.request import urlretrieve

url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
zone_file = './data/taxi_zone_lookup.csv'
urlretrieve(url, zone_file)

# 2. Read the zone lookup data
df_zones = spark.read \
    .option("header", "true") \
    .csv(zone_file)

In [64]:
df_zones.createOrReplaceTempView('zones')
dfre.createOrReplaceTempView('trips')

In [76]:
spark.sql("""
    SELECT 
        z.Zone, 
        COUNT(*) as trip_count
    FROM trips AS t
    INNER JOIN zones AS z
        ON t.PULocationID = z.LocationID
    GROUP BY 
        z.Zone
    ORDER BY trip_count ASC
""").show(truncate=False)

+---------------------------------------------+----------+
|Zone                                         |trip_count|
+---------------------------------------------+----------+
|Eltingville/Annadale/Prince's Bay            |1         |
|Governor's Island/Ellis Island/Liberty Island|1         |
|Arden Heights                                |1         |
|Port Richmond                                |3         |
|Rossville/Woodrow                            |4         |
|Rikers Island                                |4         |
|Great Kills                                  |4         |
|Green-Wood Cemetery                          |4         |
|Jamaica Bay                                  |5         |
|Westerleigh                                  |12        |
|Crotona Park                                 |14        |
|Oakwood                                      |14        |
|West Brighton                                |14        |
|New Dorp/Midland Beach                       |14       